# Clase 19 · Notebook 02
# La neurona recurrente: memoria en secuencias

**Introducción al Deep Learning — Módulo 4, Unidad 1: Redes recurrentes (Parte I)**

Una red normal (densa) mira toda la entrada **de golpe** y la trata como un conjunto de
números sin orden: es como una **bolsa de valores**. Pero en los datos **secuenciales**
(texto, series temporales, audio) el **orden importa** y a menudo hay que relacionar un
elemento con el que tiene **al lado**.

Una **neurona recurrente** resuelve esto: recorre la secuencia **paso a paso** y en cada paso
guarda un **estado oculto** (su memoria), que combina con la siguiente entrada. Así puede
recordar lo que acaba de ver.

En este cuaderno lo demostramos con un experimento sencillo: una tarea que **depende del
orden** y que una red sin memoria **no** resuelve bien, pero una recurrente **sí**.

> Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Una tarea que depende del orden

Generamos secuencias de números aleatorios. La **etiqueta es 1** si en algún punto de la
secuencia aparecen **dos valores grandes (> 1) seguidos**; y **0** en caso contrario.

Lo importante: no basta con mirar *cuántos* valores grandes hay, sino si están **juntos**.
Es decir, la respuesta depende de la **adyacencia**, y por tanto del **orden**. Para acertar,
al ver cada número hay que **recordar** si el anterior también era grande.

In [1]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
tf.random.set_seed(42); np.random.seed(42)

N = 4000          # numero de secuencias
LONG = 20         # longitud de cada secuencia
X = np.random.randn(N, LONG, 1).astype("float32")            # (secuencias, pasos, 1 valor)
# etiqueta = 1 si hay dos valores > 1 en posiciones CONSECUTIVAS
y = np.any((X[:, :-1, 0] > 1.0) & (X[:, 1:, 0] > 1.0), axis=1).astype("float32")

X_train, y_train = X[:3200], y[:3200]
X_test,  y_test  = X[3200:], y[3200:]

print("Forma de X:", X.shape, "-> (4000 secuencias, 20 pasos, 1 valor)")
print("Proporcion de etiquetas positivas: {:.0%}".format(y.mean()))
print("Acierto de 'adivinar siempre la clase mayoritaria': {:.0%}".format(max(y.mean(), 1 - y.mean())))

I0000 00:00:1784198683.999101       6 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1784198684.894359       6 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Forma de X: (4000, 20, 1) -> (4000 secuencias, 20 pasos, 1 valor)
Proporcion de etiquetas positivas: 34%
Acierto de 'adivinar siempre la clase mayoritaria': 66%


Fíjate en el último número: si siempre respondiéramos "no hay pico doble" acertaríamos
en torno al **66%** solo por la proporción de clases. Ese es el listón que hay que superar
**de verdad**.

## 2. Intento 1: una red SIN memoria

Aplanamos la secuencia y usamos una red densa normal. Al mezclar todos los pasos de golpe,
pierde la noción de **quién va al lado de quién**: ve una bolsa de 20 números sueltos y le
cuesta mucho detectar la **adyacencia**.

In [2]:
densa = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(LONG, 1)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
densa.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
densa.fit(X_train, y_train, epochs=15, batch_size=32, verbose=0)
acc_densa = densa.evaluate(X_test, y_test, verbose=0)[1]
print("Precision de la red densa (sin memoria): {:.2%}".format(acc_densa))

Precision de la red densa (sin memoria): 75.38%


## 3. Intento 2: una neurona recurrente (`SimpleRNN`)

La capa `SimpleRNN` recorre la secuencia **paso a paso** manteniendo un **estado oculto**.
En cada paso combina el valor actual con lo que recuerda del pasado inmediato. Eso le
permite responder a la pregunta clave: *el valor de ahora es grande... ¿el anterior también
lo era?*. La adyacencia deja de ser un problema.

In [3]:
rnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(LONG, 1)),
    tf.keras.layers.SimpleRNN(24),        # neurona recurrente con estado oculto
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
rnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
rnn.fit(X_train, y_train, epochs=15, batch_size=32, verbose=0)
acc_rnn = rnn.evaluate(X_test, y_test, verbose=0)[1]
print("Precision de la SimpleRNN (con memoria): {:.2%}".format(acc_rnn))
print()
print("Red densa:  {:.2%}  (muy por detrás de las recurrentes)".format(acc_densa))
print("SimpleRNN:  {:.2%}  <- recorrer la secuencia y recordar marca la diferencia".format(acc_rnn))

Precision de la SimpleRNN (con memoria): 81.75%

Red densa:  75.38%  (muy por detrás de las recurrentes)
SimpleRNN:  81.75%  <- recorrer la secuencia y recordar marca la diferencia


## 4. LSTM y GRU: memoria con puertas

La `SimpleRNN` funciona, pero cuando las secuencias son **largas** sufre el problema del
**desvanecimiento del gradiente**: la información se diluye paso a paso y la red olvida lo
que vio hace mucho.

Para resolverlo se inventaron dos variantes con **puertas** (*gates*), pequeñas compuertas
que aprenden qué información **guardar**, qué **olvidar** y qué **dejar pasar**:

- **LSTM** (*Long Short-Term Memory*): añade una **celda de memoria** y tres puertas (entrada,
  olvido y salida). Es la más potente y la más usada.
- **GRU** (*Gated Recurrent Unit*): una versión más simple y rápida, con dos puertas. A menudo
  rinde parecido a la LSTM con menos parámetros.

En Keras se usan **exactamente igual** que la `SimpleRNN`: basta con cambiar la capa.

In [4]:
resultados = {"densa": acc_densa, "SimpleRNN": acc_rnn}
for nombre, Capa in [("LSTM", tf.keras.layers.LSTM), ("GRU", tf.keras.layers.GRU)]:
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(LONG, 1)),
        Capa(24),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    modelo.fit(X_train, y_train, epochs=15, batch_size=32, verbose=0)
    resultados[nombre] = modelo.evaluate(X_test, y_test, verbose=0)[1]
    print("{:4s} -> precision: {:.2%}  ({} parametros)".format(nombre, resultados[nombre], modelo.count_params()))

print()
print("=== Resumen ===")
for k, v in resultados.items():
    print("  {:10s}: {:.2%}".format(k, v))

LSTM -> precision: 88.63%  (2521 parametros)


GRU  -> precision: 93.37%  (1969 parametros)

=== Resumen ===
  densa     : 75.38%
  SimpleRNN : 81.75%
  LSTM      : 88.63%
  GRU       : 93.37%


### ¿Qué acabamos de ver?

- La **red densa** se queda muy por detrás: al aplanar la secuencia pierde el **orden** y no
  puede detectar valores grandes **consecutivos**.
- Las redes **recurrentes** (SimpleRNN, LSTM, GRU) suben claramente porque procesan la
  secuencia paso a paso y **recuerdan** el elemento anterior.
- Las variantes con **puertas** (LSTM/GRU) son las que mejor gestionan la memoria,
  especialmente cuando las secuencias se alargan.

(Los porcentajes exactos varían un poco entre ejecuciones por la aleatoriedad del
entrenamiento, pero el patrón se mantiene: **la memoria gana**.)

## Resumen

- Los datos **secuenciales** necesitan **memoria**: el orden importa.
- Una red **densa** trata la entrada como una **bolsa de valores** sin orden y falla en
  tareas que dependen de la secuencia.
- Una **neurona recurrente** mantiene un **estado oculto** que transporta información de un
  paso al siguiente.
- La `SimpleRNN` olvida en secuencias largas (gradiente que se desvanece). **LSTM** y **GRU**
  añaden **puertas** para controlar la memoria y resolver dependencias a largo plazo.

En la **Parte II** de esta unidad daremos el salto al lenguaje real: **encoder-decoder**,
mecanismos de **atención** y **transformers**, y entrenaremos una RNN para **análisis de
sentimiento**.